In [1]:
!pip uninstall -y sparkmagic

In [2]:
!pip install -r require_phase1_20250316.txt -q

In [5]:
import os
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Uncomment to suppress TensorFlow warnings if needed

import faiss
import numpy as np
import logging
import ipywidgets as widgets
import boto3
import json
import time
import threading
from concurrent.futures import ThreadPoolExecutor
from threading import Lock
from typing import List
from IPython.display import display
from functools import lru_cache
from sentence_transformers import SentenceTransformer
import unicodedata
import re

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Configuration Dictionary (Only Selected Models)
CONFIG = {
    "bucket_name": "sagemaker-us-east-1-188494237500",
    "json_folder": "dev/json",
    "index_folder": "dev/idx",
    "nlist": 512,
    "nprobe": 16,
    "num_training_samples": 1000,
    "embedding_models": {
        "amazon.titan-embed-text-v2:0": 1536,
        "anthropic.claude-3-sonnet-20240229-v1:0": 8192,
        "meta.llama3-70b-instruct-v1:0": 8192
    },
    "default_model": "amazon.titan-embed-text-v2:0"
}

# Initialize S3 and Bedrock clients
s3_client = boto3.client("s3")
bedrock = boto3.client("bedrock-runtime")
query_cache = {}
faiss_lock = Lock()  # Lock for FAISS index updates

In [6]:
# This function requires ObjectTagging Read/Write permissions
def check_s3_object_tag(bucket: str, key: str, tag_key: str) -> bool:
    """Checks if an object in S3 has a specific tag."""
    try:
        response = s3_client.get_object_tagging(Bucket=bucket, Key=key)
        tags = {tag["Key"]: tag["Value"] for tag in response.get("TagSet", [])}
        return tag_key in tags
    except s3_client.exceptions.ClientError as e:
        if e.response['Error']['Code'] == 'AccessDenied':
            logging.warning(f"Skipping tag check for {key} due to insufficient permissions.")
            return False  # Assume the object has no tag and continue
        else:
            logging.error(f"Error retrieving tags for {key}: {e}")
            return False

def list_objects_s3(bucket: str, prefix: str) -> List[str]:
    """Lists all objects in an S3 bucket under a given prefix."""
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
        return [obj["Key"] for obj in response.get("Contents", [])]
    except Exception as e:
        logging.error(f"⚠️ Error listing S3 objects: {e}")
        return []

# Load FAISS index from S3
def load_faiss_index(model_name):
    bucket = CONFIG["bucket_name"]
    index_key = f"{CONFIG['index_folder']}/{model_name}.faiss"
    
    try:
        response = s3_client.get_object(Bucket=bucket, Key=index_key)
        index_data = np.frombuffer(response["Body"].read(), dtype=np.uint8)
        index = faiss.deserialize_index(index_data)
        logger.info(f"✅ Loaded FAISS index for {model_name} from S3.")
    except s3_client.exceptions.NoSuchKey:
        logger.info(f"⚠️ No existing FAISS index found for {model_name}, creating a new one. This may take some time...")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])
    except Exception as e:
        logger.error(f"❌ Error loading FAISS index for {model_name}: {e}. Creating a new one.")
        index = faiss.IndexFlatL2(CONFIG["embedding_models"][model_name])
    
    return index, index_key

# Save FAISS index to S3
def save_faiss_index(index, index_key):
    with faiss_lock:  # Prevent multiple threads from writing at the same time
        try:
            index_data = faiss.serialize_index(index).tobytes()
            s3_client.put_object(Bucket=CONFIG["bucket_name"], Key=index_key, Body=index_data)
            logger.info(f"✅ FAISS index saved to {index_key}")
        except Exception as e:
            logger.error(f"❌ Failed to save FAISS index to {index_key}: {e}")

# Clean text function
def clean_text(text: str) -> str:
    """
    Cleans extracted text by normalizing Unicode characters,
    removing extra newlines, and collapsing multiple whitespace characters.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.replace('\n', ' ').replace('\t', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Bedrock Embedding Function with Cache and document-level embedding
def generate_embedding(text, model_name):
    """
    Generates an embedding for the given text using AWS Bedrock.
    - Checks the cache to avoid redundant API calls.
    - Handles missing or malformed responses correctly.
    """
    cache_key = (text, model_name)
    
    if cache_key in query_cache:
        logger.info(f"⚡ Using cached embedding for '{text}' ({model_name})")
        return query_cache[cache_key]

    try:
        logger.info(f"⚡ Requesting embedding from AWS Bedrock for {model_name}...")
        start_time = time.time()

        # ✅ Make the request
        response = bedrock.invoke_model(
            modelId=model_name,
            contentType="application/json",
            accept="application/json",
            body=json.dumps({"inputText": text})
        )

        elapsed = time.time() - start_time
        logger.info(f"🕒 Bedrock embedding request took {elapsed:.2f} seconds.")

        # ✅ Validate response structure
        if not response or "Body" not in response:
            logger.error(f"❌ AWS Bedrock response for {model_name} is missing 'Body'. Full response: {response}")
            return []

        # ✅ Read and decode only once
        response_body_str = response["Body"].read()
        if not response_body_str:
            logger.error(f"❌ Empty response body for {model_name}. Skipping embedding generation.")
            return []

        response_body = json.loads(response_body_str.decode("utf-8"))

        # ✅ Ensure 'embedding' exists in response
        if "embedding" not in response_body:
            logger.error(f"❌ 'embedding' key missing in Bedrock response for {model_name}. Full response: {response_body}")
            return []

        embedding = response_body["embedding"]

    except json.JSONDecodeError as e:
        logger.error(f"❌ JSON decoding error for {model_name}: {e}")
        return []
    except Exception as e:
        logger.error(f"❌ Unexpected error generating embedding for {model_name}: {e}")
        return []

    # ✅ Ensure embedding dimension matches expected value
    expected_dim = CONFIG["embedding_models"].get(model_name, None)
    if expected_dim and len(embedding) != expected_dim:
        logger.warning(f"⚠️ Embedding dimension mismatch for {model_name}: Expected {expected_dim}, got {len(embedding)}")
        return []

    # ✅ Store in cache
    query_cache[cache_key] = embedding
    return embedding


def process_single_file(obj, model_name):
    """
    Processes a single JSON file from S3:
    - Reads the file.
    - Combines text from all chunks to create a single document-level text.
    - Generates an embedding for the combined text.
    - Updates the JSON file with the new embedding.
    """
    bucket = CONFIG["bucket_name"]
    file_key = obj["Key"]
    
    try:
        file_obj = s3_client.get_object(Bucket=bucket, Key=file_key)
        file_content_str = file_obj["Body"].read().decode("utf-8").strip()
        if not file_content_str:
            logger.warning(f"⚠️ Empty JSON file: {file_key}. Skipping.")
            return
        file_content = json.loads(file_content_str)
    except json.JSONDecodeError:
        logger.error(f"❌ Invalid JSON format in {file_key}. Skipping.")
        return
    except Exception as e:
        logger.error(f"❌ Failed to load {file_key}: {e}")
        return
    
    # Combine all text from chunks to generate a single document-level embedding
    combined_text = " ".join(clean_text(chunk.get("text", "").strip()) for chunk in file_content.get("chunks", []))
    
    if combined_text:
        if "embeddings" not in file_content:
            file_content["embeddings"] = {}
        embedding_result = generate_embedding(combined_text, model_name)
        if not embedding_result:
            logger.error(f"❌ No embedding generated for {file_key}. Skipping write.")
            return
        file_content["embeddings"][model_name] = embedding_result
    
    json_body = json.dumps(file_content, ensure_ascii=False).encode("utf-8")
    s3_client.put_object(Bucket=bucket, Key=file_key, Body=json_body)
    logger.info(f"✅ Embeddings successfully saved in {file_key}")
    if "embeddings" not in file_content or model_name not in file_content["embeddings"]:
        logger.error(f"❌ Embeddings were NOT saved in {file_key}. Check AWS Bedrock response.")
    else:
        logger.info(f"✅ Updated embeddings in {file_key}")

def process_json_files(model_name):
    """
    Processes all JSON files in the configured S3 folder:
    - Loads or creates a FAISS index.
    - Retrieves a list of JSON files.
    - Processes each file in parallel to generate document-level embeddings.
    - Saves the FAISS index back to S3.
    """
    start_time = time.time()
    logger.info(f"🚀 Starting processing for model {model_name}...")
    bucket = CONFIG["bucket_name"]
    folder = CONFIG["json_folder"]
    index, index_key = load_faiss_index(model_name)
    
    logger.info("🔍 Retrieving file list from S3...")
    start_list_time = time.time()
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix=folder)
    logger.info(f"📌 S3 File List Retrieved in {time.time() - start_list_time:.2f} sec")
    if "Contents" not in response or len(response["Contents"]) == 0:
        logger.error("❌ No JSON files found in S3 bucket. Ensure the folder is populated. No processing was performed.")
        return
    
    with ThreadPoolExecutor(max_workers=5) as executor:
        executor.map(process_single_file, response["Contents"], [model_name] * len(response["Contents"]))
    
    save_faiss_index(index, index_key)
    duration = time.time() - start_time
    logger.info(f"Finished processing all JSON files in {duration:.2f} seconds. Total files processed: {len(response['Contents'])}")

# UI Setup: Create widgets for model selection and execution.
model_selector = widgets.Dropdown(
    options=list(CONFIG["embedding_models"].keys()),
    value=CONFIG["default_model"],
    description="Embedding Model:",
)
execute_button = widgets.Button(
    description="Execute",
    button_style="success"
)
# When the Execute button is clicked, run processing in a background thread.
execute_button.on_click(lambda _: threading.Thread(target=process_json_files, args=(model_selector.value,)).start())

# Display the UI immediately
display(model_selector, execute_button)

# ==============================================
# END OF SCRIPT
# ==============================================

Dropdown(description='Embedding Model:', options=('amazon.titan-embed-text-v2:0', 'anthropic.claude-3-sonnet-2…

Button(button_style='success', description='Execute', style=ButtonStyle())

INFO:__main__:🚀 Starting processing for model amazon.titan-embed-text-v2:0...
INFO:__main__:✅ Loaded FAISS index for amazon.titan-embed-text-v2:0 from S3.
INFO:__main__:🔍 Retrieving file list from S3...
INFO:__main__:📌 S3 File List Retrieved in 0.12 sec
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0...
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0...
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0...
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0...
INFO:__main__:⚡ Requesting embedding from AWS Bedrock for amazon.titan-embed-text-v2:0...
INFO:__main__:🕒 Bedrock embedding request took 0.13 seconds.
ERROR:__main__:❌ AWS Bedrock response for amazon.titan-embed-text-v2:0 is missing 'Body'. Full response: {'ResponseMetadata': {'RequestId': 'd3059071-cb0c-4303-934a-4d3e1ee5df4e', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Mon, 17